In [30]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import sys
import json
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [31]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [32]:
load_dotenv()
src_path = os.getenv("SRC_PATH")
data_path = os.getenv("DATA_PATH")
transaction_path = os.path.join(data_path, r'raw/train_transaction.csv/train_transaction.csv')
identity_path = os.path.join(data_path, r'raw/train_identity.csv/train_identity.csv')
train_transaction = pd.read_csv(transaction_path)
train_identity = pd.read_csv(identity_path)
full_df  = train_transaction.merge(train_identity, on="TransactionID", how = "left")
full_df  = df.sort_values("TransactionDT").reset_index(drop=True)

In [33]:
sys.path.append(src_path)
from features.engineering import create_engineered_features
from data.split import temporal_split

In [34]:
json_path = os.path.join(data_path, "processed", "split_info.json")
with open(json_path, 'r') as f:
    split_info = json.load(f)

train_end = split_info.get("train_end")
val_end = split_info.get("validation_end")

In [35]:
train_df, val_df, test_df = temporal_split(full_df, train_end, val_end)
y_datasets = {}
map_dfs = {"train": train_df, "val": val_df, "test": test_df}

for name, subset_df in map_dfs.items():
    y_datasets[f'y_{name}'] = subset_df['isFraud']

In [36]:
def add_amount_features(df):
    df = df.copy()

    df["transaction_amt_log"] = np.log1p(df["TransactionAmt"])

    df["amount_decimal"] = (df["TransactionAmt"] % 1)

    return df

In [37]:
def create_d_time_features(df):
    df = df.copy()

    df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)

    df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)

    return df

In [38]:
def add_email_features(df):
    df = df.copy()

    for col in ["P_emaildomain", "R_emaildomain"]:
        df[f'{col}_is_missing'] = df[col].isna().astype(int)

        df[f"{col}_provider"] = df[col].fillna("missing").str.split(".").str[0]

    return df

In [39]:
def create_d_features(df):
    df = df.copy()

    df = create_engineered_features(df)

    df = add_amount_features(df)

    df = create_d_time_features(df)

    df = add_email_features(df)

    return df

In [40]:
d_features_dfs = {}

d_features_dfs.clear()
for name, subset_df in map_dfs.items():
    d_features_dfs[f'{name}'] = create_d_features(subset_df)

In [41]:
assert list(d_features_dfs['train'].columns) == list(d_features_dfs['val'].columns)
assert list(d_features_dfs['train']) == list(d_features_dfs['test'].columns)

In [42]:
def get_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()

    num_cols = X.select_dtypes(include=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))])


    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)])

    return preprocessor

def create_pipeline(X):
    preprocessor = get_preprocessor(X)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear"))])

    return pipe

def evaluate_model(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:,1]

    return {"pr_auc": average_precision_score(y, probabilities),
            "roc_auc": roc_auc_score(y, probabilities),
            "precision": precision_score(y, predictions, zero_division=0),
            "recall": recall_score(y, predictions, zero_division=0),
            "f1": f1_score(y, predictions, zero_division=0)
            }

In [43]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name="fraud-detection-baseline")
result = []
with mlflow.start_run(run_name="Logistic_regression_D_features"):
    X_train = d_features_dfs['train']
    pipe = create_pipeline(X_train)

    pipe.fit(X_train, y_datasets['y_train'])

    metrics = evaluate_model(pipe, d_features_dfs['val'], y_datasets['y_val'])

    mlflow.log_param("model", "logistic_regression")

    mlflow.log_param("class_weight", "balanced")

    mlflow.log_param("feature_count", X_train.shape[1])

    mlflow.log_metrics(metrics)

    result.append({**metrics})

🏃 View run Logistic_regression_D_features at: http://127.0.0.1:5000/#/experiments/1/runs/506bc427958b4e4995ab68a53311fb6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


ValueError: Found array with 0 sample(s) (shape=(0, 22)) while a minimum of 1 is required by SimpleImputer.

In [46]:
y_datasets['y_train'].shape

(0,)

In [ ]:
assert len(d_features_dfs["train"]) == len(y_datasets["y_train"]), "Row count mismatch!"